In [17]:
#@title Global configuration
PROJECT_ID      = "qwiklabs-gcp-02-e659e41ff1eb"          # change
LOCATION        = "us"                      # keep consistent across services
DATASET_ID      = "airport_weather"
AIRPORT_TABLE   = "us_airports"
FORECAST_TABLE  = "airport_forecasts_raw"
ALERT_TABLE     = "airport_alerts"
MODEL_ID        = "gemini_flash"
GCS_URI         = "gs://labs.roitraining.com/data-to-ai-workshop/airports.csv"

import os, json, requests, pandas as pd, google.cloud.bigquery as bq
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
bq_client = bq.Client(project=PROJECT_ID, location=LOCATION)
print("Environment ready ✅")

Environment ready ✅


In [18]:
dataset_ref = bq.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
bq_client.create_dataset(dataset_ref, exists_ok=True)

job_cfg = bq.LoadJobConfig(
    source_format=bq.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)
bq_client.load_table_from_uri(
    GCS_URI,
    f"{PROJECT_ID}.{DATASET_ID}.{AIRPORT_TABLE}",
    job_config=job_cfg).result()

print("Airports table loaded 🚀")

Airports table loaded 🚀


In [19]:
large_q = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{AIRPORT_TABLE}` AS
SELECT
  ident, name, iso_country, iso_region,
  latitude_deg  AS lat,
  longitude_deg AS lon
FROM `{PROJECT_ID}.{DATASET_ID}.{AIRPORT_TABLE}`
WHERE type = 'large_airport';
"""
bq_client.query(large_q).result()
print("Retained only large airports ✈️")

Retained only large airports ✈️


In [20]:
forecast_rows = []

airports_df = bq_client.query(f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{AIRPORT_TABLE}`").to_dataframe()

for _, row in airports_df.iterrows():
    try:
        # 2-step NOAA flow
        meta = requests.get(
            f"https://api.weather.gov/points/{row.lat},{row.lon}",
            timeout=10,
            headers={"User-Agent": "ai-weather-lab"}
        ).json()
        forecast_url = meta["properties"]["forecast"]
        forecast_json = requests.get(forecast_url, timeout=10, headers={"User-Agent": "ai-weather-lab"}).json()
        # Grab first 2 periods (today + tonight)
        periods = forecast_json["properties"]["periods"][:2]
        fc_text = " ".join(p["detailedForecast"] for p in periods)

        forecast_rows.append({
            "ident": row.ident,
            "forecast_retrieved": pd.Timestamp.utcnow(),
            "raw_forecast": fc_text
        })
    except Exception as ex:
        print(f"⚠️  {row.ident}: {ex}")

forecast_df = pd.DataFrame(forecast_rows)
print(f"Collected {len(forecast_df)} forecasts")

⚠️  OMAA: 'properties'
⚠️  OMDW: 'properties'
⚠️  OMDB: 'properties'
⚠️  OMSJ: 'properties'
⚠️  LATI: 'properties'
⚠️  UDYZ: 'properties'
⚠️  FNLU: 'properties'
⚠️  SAEZ: 'properties'
⚠️  SABE: 'properties'
⚠️  LOWW: 'properties'
⚠️  YSSY: 'properties'
⚠️  YPDN: 'properties'
⚠️  YBBN: 'properties'
⚠️  YPAD: 'properties'
⚠️  YMML: 'properties'
⚠️  YPPH: 'properties'
⚠️  TNCA: 'properties'
⚠️  UBBB: 'properties'
⚠️  LQSA: 'properties'
⚠️  VGHS: 'properties'
⚠️  EBBR: 'properties'
⚠️  LBBG: 'properties'
⚠️  LBWN: 'properties'
⚠️  LBSF: 'properties'
⚠️  OBBI: 'properties'
⚠️  WBSB: 'properties'
⚠️  SLVR: 'properties'
⚠️  TNCB: 'properties'
⚠️  SBEG: 'properties'
⚠️  SBSV: 'properties'
⚠️  SBPS: 'properties'
⚠️  SBFZ: 'properties'
⚠️  SBBR: 'properties'
⚠️  SBVT: 'properties'
⚠️  SBCF: 'properties'
⚠️  SBBE: 'properties'
⚠️  SBGL: 'properties'
⚠️  SBFL: 'properties'
⚠️  SBGR: 'properties'
⚠️  MYNN: 'properties'
⚠️  FBSK: 'properties'
⚠️  UMMS: 'properties'
⚠️  MZBZ: 'properties'
⚠️  CYYC: '

In [21]:
job_cfg = bq.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
bq_client.load_table_from_dataframe(
    forecast_df,
    f"{PROJECT_ID}.{DATASET_ID}.{FORECAST_TABLE}",
    job_config=job_cfg).result()
print("Forecasts loaded 🗄️")

Forecasts loaded 🗄️


In [22]:
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`REMOTE
WITH CONNECTION DEFAULT
OPTIONS(
  ENDPOINT  = 'gemini-2.5-flash'
);
"""
bq_client.query(create_model_sql).result()
print("Gemini Flash model registered 🎉")

Gemini Flash model registered 🎉


In [23]:
alert_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{ALERT_TABLE}` AS
SELECT
  ident,
  raw_forecast,
  ml_generate_text_result AS alert_text,
  CURRENT_TIMESTAMP() AS alert_ts
FROM ML.GENERATE_TEXT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`,
  (
    SELECT
      ident,
      raw_forecast,
      CONCAT(
        'You are an aviation weather assistant. ',
        'Create a concise alert for pilots (<60 words) based on this forecast: ',
        raw_forecast
      ) AS prompt      -- the column name must literally be 'prompt'
    FROM `{PROJECT_ID}.{DATASET_ID}.{FORECAST_TABLE}`
  ),
  STRUCT(
    0.3  AS temperature,
    120  AS max_output_tokens
  )
);
"""
bq_client.query(alert_sql).result()
print("Alerts table created without prompt_response_id join ✅")

Alerts table created without prompt_response_id join ✅


In [24]:
bq_client.query(f"""
SELECT ident, alert_text
FROM `{PROJECT_ID}.{DATASET_ID}.{ALERT_TABLE}`
LIMIT 5
""").to_dataframe()

,ident,alert_text
0,PANC,"{""candidates"":[{""avg_logprobs"":-0.349933415651..."
1,KIAH,"{""candidates"":[{""avg_logprobs"":-0.228469971248..."
2,KIND,"{""candidates"":[{""avg_logprobs"":-0.306172897941..."
3,KLAS,"{""candidates"":[{""avg_logprobs"":-0.339075160026..."
4,KCLE,"{""candidates"":[{""avg_logprobs"":-0.327476059518..."
